# Stage 2 neural comparison (Google Colab)

This thin runner clones the committed repository, installs its declared requirements, runs the configured Stage 2 search, and copies result CSV/JSON files to Google Drive. Select a GPU runtime in **Runtime → Change runtime type** before running all cells. The experiment itself is the package CLI; no model logic is duplicated in this notebook.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import subprocess
import tempfile
from pathlib import Path

repository_url = 'https://github.com/Nyamekye-tech/gru-mixed-adc-channel-estimator.git'
repository_path = Path(tempfile.mkdtemp(prefix='chest-stage2-'))
subprocess.run(
    ['git', 'clone', '--depth', '1', repository_url, str(repository_path)],
    check=True,
)
print(f'Cloned into {repository_path}')

In [ ]:
import sys
import torch

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'],
    cwd=repository_path,
    check=True,
)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('Select a GPU runtime before running the neural search.')

In [ ]:
from datetime import datetime, timezone
import shutil

run = subprocess.run(
    [
        sys.executable,
        '-m',
        'experiments.run_stage2',
        '--config',
        'configs/stage2.yaml',
    ],
    cwd=repository_path,
)
commit = subprocess.run(
    ['git', 'rev-parse', 'HEAD'],
    cwd=repository_path,
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()

timestamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
drive_output = (
    Path('/content/drive/MyDrive/channel-estimator/stage2')
    / f'{commit[:12]}_{timestamp}'
)
drive_output.mkdir(parents=True, exist_ok=False)
result_names = (
    'stage2_neural_results.csv',
    'stage2_neural_search.csv',
    'stage2_neural_metadata.json',
)
for name in result_names:
    source = repository_path / 'results' / name
    if source.is_file():
        shutil.copy2(source, drive_output / name)
print(f'Results and any diagnostic outputs copied to {drive_output}')
if run.returncode != 0:
    raise RuntimeError(
        f'Stage 2 exited with status {run.returncode}; inspect the copied '
        'metadata and comparison-guard failures before reporting results.'
    )